In [5]:
import sys
sys.path.append('/home/caoren/tmp/PIRC_for_HigherOrderCausality')
from Model.this import *
import os
os.chdir("/home/caoren/tmp/PIRC_for_HigherOrderCausality/Results2/EEG")
import numpy as np
from Torch_Library import *
from PIRC import PIRC_flatten as Nonliear_PIRC
from Reconstruction_PIRC import *
import numpy as np
from itertools import combinations
import argparse
import math
import numpy as np
import matplotlib.pyplot as plt
import pickle
from collections import Counter
import optuna
optuna.logging.set_verbosity(optuna.logging.CRITICAL)

In [ ]:
# =========================
# Tools
# =========================
def Causal_score(y_pred, y_true, method="exp", tau=2, eps=1e-12):

    # 转成 tensor
    y_pred = torch.as_tensor(y_pred)
    y_true = torch.as_tensor(y_true)

    # mse over time dimension only
    mse = (y_pred - y_true).pow(2).mean(dim=-1).abs()         # (...,)

    # scale: mean over time of y_true (同你原实现)
    # scale = y_true.mean(dim=-1).abs().clamp_min(eps)          # (...,)
    scale1 = y_true.pow(2).mean(dim=-1).abs()
    scale2 = y_true.var(dim=-1, correction=0)# (...,)

    return torch.sqrt(mse / scale1)
    return torch.sqrt(mse / scale2)
def build_T_to_Ainf(T_from_Ainf, max_order=3):
    T_to_Ainf = {o: [] for o in range(2, max_order + 1)}

    row, col = T_from_Ainf.shape
    node_num = row

    for i in range(row):
        for j in range(col):
            val = T_from_Ainf[i, j]

            if val == 0:
                continue

            if j < node_num:
                if j != i:
                    T_to_Ainf[2].append([i, j, val])

            elif j < node_num + math.comb(node_num - 1, 2):
                idx = j - node_num
                others = [x for x in range(node_num) if x != i]
                comb_list = list(combinations(others, 2))
                j0, k0 = comb_list[idx]
                T_to_Ainf[3].append([i, j0, k0, val])

            elif max_order == 4:
                idx = j - node_num - math.comb(node_num - 1, 2)
                others = [x for x in range(node_num) if x != i]
                comb_list = list(combinations(others, 3))
                j0, k0, l0 = comb_list[idx]
                T_to_Ainf[4].append([i, j0, k0, l0, val])

    for o in T_to_Ainf:
        T_to_Ainf[o] = (
            np.array(T_to_Ainf[o])
            if len(T_to_Ainf[o]) > 0
            else np.zeros((0, o + 1))
        )

    return T_to_Ainf


def top_freq_per_sequence(Ainf_list, order=3, top_n=5, threshold=None):
    """
    每个序列：
    1. 可选：剔除 |weight| < threshold 的边
    2. 按 |weight| 从大到小排序
    3. 取 top_n
    4. 统计超边出现频率
    """
    all_top_edges = []

    for Ainf in Ainf_list:
        if order not in Ainf or Ainf[order].shape[0] == 0:
            continue

        edges = Ainf[order]
        weights = np.abs(edges[:, -1])

        if threshold is not None:
            valid = weights >= threshold
            edges = edges[valid]
            weights = weights[valid]

        if len(weights) == 0:
            continue

        k = min(top_n, len(weights))
        idx_top = np.argsort(weights)[-k:][::-1]

        top_edges = edges[idx_top, :order]

        for row in top_edges:
            row = row.astype(int)

            if order == 3:
                target = int(row[0])
                j, k0 = sorted([int(row[1]), int(row[2])])
                edge = (target, j, k0)
            else:
                edge = tuple(row)

            all_top_edges.append(edge)

    return Counter(all_top_edges)


def high_order_ratio(Score, n=7):
    if torch.is_tensor(Score):
        Score = Score.detach().cpu().numpy()

    Score = np.asarray(Score).copy()

    for i in range(n):
        Score[i, i] = 0.0

    higher = Score[:, n:].sum()
    total = Score.sum()

    return higher / (total + 1e-12)


def normalize_data(X0, norm):
    if norm == 1:
        scale = np.mean(np.abs(X0))
        if scale < 1e-12:
            return None
        return X0 / scale

    elif norm == 2:
        return (X0 - X0.mean(axis=1, keepdims=True)) / (
            X0.std(axis=1, keepdims=True) + 1e-12
        )

    else:
        raise ValueError("norm must be 1 or 2")


# =========================
# Main search
# =========================

device = torch.device("cuda:2")

nz = 7
subjects = list_all_subjects(109)
states = ["01", "02"]

s = np.loadtxt(
    f"../../EEG_data/eeg-data/sensors-{nz}.csv",
    dtype=str,
    delimiter=","
)

z = np.loadtxt(
    f"../../EEG_data/eeg-data/zones-{nz}.csv",
    dtype=int,
    delimiter=","
)

s2z = {s[i]: z[i] for i in range(len(s))}

valid_results = []
all_results = []
while len(valid_results) ==0:
    for N_train in [500]:

        for I_type in [1]:

            for norm in [1,2]:

                print("\n" + "=" * 100)
                print(f"Running: N_train={N_train}, I_type={I_type}, norm={norm}")
                print("=" * 100)

                args = argparse.Namespace(
                    dt=1 / 160,
                    n=7,
                    N_train=N_train,
                    N_test=10,
                    N_washout=0,
                    N_start=0,
                    norm=norm,
                    I_type=I_type,
                    n_trials=100
                )

                # ========= hyperparameter search =========
                params, Win, Wres = gridSearch_PIRC(args)

                n_units = params["n_units"]
                alpha = params["alpha"]
                sigma_in = params["sigma_in"]
                rho = params["rho"]
                tikh = params["tikh"]
                option = params["option"]
                tau = params["tau"]
                method = params["method"]
                connectivity = params["connectivity"]
                mode = params["mode"]
                block_dim = params["block_dim"]
                bias = params["bias"]

                Win = torch.as_tensor(Win, dtype=torch.float32, device=device)
                Wres = torch.as_tensor(Wres, dtype=torch.float32, device=device)

                T_need = (
                    args.N_train
                    + args.N_test
                    + args.N_washout
                    + args.N_start
                    + 1
                )

                Loss = []
                Score_all = []
                Score_01 = []
                Score_02 = []

                # ========= evaluate all subjects =========
                for su in subjects:

                    for st in states:

                        key = f"S{su}R{st}"
                        file = (
                            f"../../EEG_data/physionet.org/files/eegmmidb/1.0.0/"
                            f"S{su}/{key}.edf"
                        )

                        try:
                            s2signal = read_eeg(file)
                        except Exception as e:
                            print(f"[Skip] {key}: read error: {e}")
                            continue

                        asig = average_over_zones(s2signal, s2z)

                        tmp = np.max(np.abs(asig), axis=0)
                        valid_idx = np.where(tmp > 1e-6)[0]

                        if len(valid_idx) == 0:
                            print(f"[Skip] {key}: no valid signal")
                            continue

                        start, end = valid_idx[0], valid_idx[-1]

                        X0 = denoise_fourier(asig[:, start:end + 1], 100)

                        if X0.shape[1] < T_need:
                            print(
                                f"[Skip] {key}: length {X0.shape[1]} < T_need {T_need}"
                            )
                            continue

                        X0 = normalize_data(X0, norm=args.norm)

                        if X0 is None:
                            print(f"[Skip] {key}: normalization failed")
                            continue

                        X_np = X0[:, :T_need]

                        X = torch.as_tensor(
                            X_np,
                            device=device,
                            dtype=torch.float32
                        ).T

                        X_batch = torch.stack(
                            [shift_column_to_first(X, i) for i in range(args.n)],
                            dim=0
                        )

                        X_washout, X_train, Y_train, Y_test = split_dataset2_batch(
                            X_batch,
                            args.N_washout,
                            args.N_train,
                            args.N_test,
                            in_dim=args.n,
                            out_dim=args.n
                        )

                        pirc = Nonliear_PIRC(
                            n_units=n_units,
                            in_dim=args.n,
                            out_dim=args.n,
                            Win=Win.clone(),
                            Wres=Wres.clone(),
                            Expand=1,
                            sigma_in=sigma_in,
                            rho=rho,
                            alpha=alpha,
                            tikh=tikh,
                            mode=mode,
                            block_dim=block_dim,
                            I_type=args.I_type,
                            option=option,
                            device=device,
                            dt=args.dt,
                            bias=bias
                        )

                        R, _ = pirc.train(X_washout, X_train, Y_train)

                        Y_test_predict = pirc.Prediction2(
                            R,
                            args.N_test,
                            Y_train,
                            Y_test
                        )

                        base_all = Y_test_predict[0, :, :args.N_test, 0]
                        others_all = Y_test_predict[2:, :, :args.N_test, 0]
                        target = Y_test[:, :, 0]

                        total_mse = (base_all - target).pow(2).mean()
                        Loss.append(total_mse.item())

                        base_expand = base_all.unsqueeze(0).expand_as(others_all)

                        score_EN = Causal_score(
                            others_all,
                            base_expand,
                            method=method,
                            tau=tau
                        )

                        Score = score_EN.transpose(0, 1).contiguous()
                        Score = shift_column_for_Causal_Matrix(Score)
                        Score = Score.detach().cpu()

                        Score_all.append(Score)

                        if st == "01":
                            Score_01.append(Score)
                        else:
                            Score_02.append(Score)

                if len(Loss) == 0 or len(Score_all) == 0:
                    print("[Warning] no valid results for this setting")
                    continue

                mean_loss = float(np.mean(Loss))

                Ainf_PIRC_list = [
                    build_T_to_Ainf(
                        Score.detach().cpu().numpy()
                        if torch.is_tensor(Score)
                        else np.asarray(Score),
                        max_order=3
                    )
                    for Score in Score_all
                ]

                top_select = 10

                freq_PIRC = top_freq_per_sequence(
                    Ainf_PIRC_list,
                    order=3,
                    top_n=top_select,
                    threshold=None
                )

                top5_edges = freq_PIRC.most_common(5)

                rho_all = np.array([
                    high_order_ratio(S, n=7)
                    for S in Score_all
                ])

                rho_mean = float(rho_all.mean())
                rho_std = float(rho_all.std())

                top5_all_target0 = (
                    len(top5_edges) >= 5
                    and all(edge[0] == 0 for edge, count in top5_edges)
                )

                result = {
                    "N_train": N_train,
                    "I_type": args.I_type,
                    "norm": args.norm,
                    "params": params,
                    "mean_loss": mean_loss,
                    "rho_mean": rho_mean,
                    "rho_std": rho_std,
                    "top5_edges": top5_edges,
                    "top5_all_target0": top5_all_target0,
                    "Win": Win.detach().cpu().numpy(),
                    "Wres": Wres.detach().cpu().numpy(),
                }

                all_results.append(result)

                print(f"Average Loss: {mean_loss:.6f}")
                print(f"rho_all: {rho_mean:.4f} ± {rho_std:.4f}")
                print("Top5:", top5_edges)
                print("Top5 all target 0:", top5_all_target0)

                if rho_mean > 0.5 and top5_all_target0:
                    valid_results.append(result)
                    print("✅ FOUND valid setting!")

                # 防止显存累积
                torch.cuda.empty_cache()


# =========================
# Summary
# =========================

print("\n\n" + "=" * 100)
print("VALID RESULTS")
print("=" * 100)

if len(valid_results) == 0:
    print("No valid setting found.")
else:
    for r in valid_results:
        print(
            f"N_train={r['N_train']}, "
            f"I_type={r['I_type']}, "
            f"norm={r['norm']}, "
            f"loss={r['mean_loss']:.6f}, "
            f"rho={r['rho_mean']:.4f}±{r['rho_std']:.4f}"
        )
        print("Top5:", r["top5_edges"])
        print("Params:", r["params"])
        print("-" * 100)


# =========================
# Save
# =========================

with open("search_target0_results3.pkl", "wb") as f:
    pickle.dump(
        {
            "all_results": all_results,
            "valid_results": valid_results,
        },
        f
    )

print("Saved to search_target0_results3.pkl")


Running: N_train=500, I_type=1, norm=1
Study finished.

Best Hyperparameters: {'n_units': 10, 'alpha': 0.1, 'sigma_in': 0.6, 'rho': 1.0, 'tikh': 0.1, 'option': 3, 'tau': 2, 'method': 'Logistic', 'connectivity': 1.0, 'mode': 2, 'block_dim': 2, 'bias': 0}

Lowest Loss: 2.984062666655518e-05
Average Loss: 0.000078
rho_all: 0.6099 ± 0.1029
Top5: [((0, 2, 3), 44), ((1, 0, 3), 42), ((2, 0, 3), 42), ((0, 1, 3), 41), ((1, 0, 5), 39)]
Top5 all target 0: False

Running: N_train=500, I_type=1, norm=2
Study finished.

Best Hyperparameters: {'n_units': 10, 'alpha': 0.1, 'sigma_in': 0.9, 'rho': 1.0, 'tikh': 0.1, 'option': 3, 'tau': 2, 'method': 'Logistic', 'connectivity': 1.0, 'mode': 2, 'block_dim': 2, 'bias': 0}

Lowest Loss: 1.431459986633854e-05
Average Loss: 0.000033
rho_all: 0.5771 ± 0.1128
Top5: [((2, 4, 5), 50), ((1, 4, 5), 50), ((0, 2, 5), 44), ((0, 4, 5), 42), ((6, 3, 4), 36)]
Top5 all target 0: False

Running: N_train=500, I_type=1, norm=1
Study finished.

Best Hyperparameters: {'n_units

In [ ]:
nz = 7  # scalp zones
subjects = list_all_subjects(109)
states = ["01", "02"]  # resting
s = np.loadtxt(f"../../EEG_data/eeg-data/sensors-{nz}.csv", dtype=str, delimiter=",")
z = np.loadtxt(f"../../EEG_data/eeg-data/zones-{nz}.csv", dtype=int, delimiter=",")
s2z = {s[i]: z[i] for i in range(len(s))}
Score_all=[]
Score_01=[]
Score_02=[]
Loss=[]
for su in subjects:
    for st in states:
        key = f"S{su}R{st}"
        file = f"../../EEG_data/physionet.org/files/eegmmidb/1.0.0/S{su}/{key}.edf"
        s2signal = read_eeg(file)
        asig = average_over_zones(s2signal, s2z)

        tmp = np.max(np.abs(asig), axis=0)
        valid_idx = np.where(tmp > 1e-6)[0]
        if len(valid_idx) == 0:
            continue
        if np.all(np.diff(valid_idx) == 1):
            pass
        else:
            print("不连续")
        start, end = valid_idx[0], valid_idx[-1]

        X0 = denoise_fourier(asig[:, start:end + 1], 100)
        # X0 = X0 / np.mean(np.abs(X0))
        X = (X0 - X0.mean(axis=1, keepdims=True)) / (X0.std(axis=1, keepdims=True) + 1e-12)
        X = torch.as_tensor(X, device=device, dtype=torch.float32).T
        X_batch = torch.stack(
            [shift_column_to_first(X, i) for i in range(args.n)],
            dim=0
        )

        X_washout, X_train, Y_train, Y_test = split_dataset2_batch(
            X_batch,
            args.N_washout,
            args.N_train,
            args.N_test,
            in_dim=args.n,
            out_dim=args.n
        )

        pirc = Nonliear_PIRC(
            n_units=n_units,
            in_dim=args.n,
            out_dim=args.n,
            Win=Win,
            Wres=Wres,
            Expand=1,
            sigma_in=sigma_in,
            rho=rho,
            alpha=alpha,
            tikh=tikh,
            mode=mode,
            block_dim=block_dim,
            I_type=I_type,
            option=option,
            device=device,
            dt= 1 / 160,
            bias=bias
        )
        R, _ = pirc.train(X_washout, X_train, Y_train)
        if mode == 1:
            Y_test_predict = pirc.Prediction(R, args.N_test, Y_train, Y_test)
        else:
            Y_test_predict = pirc.Prediction2(R, args.N_test, Y_train, Y_test)
        base_all = Y_test_predict[0, :, :args.N_test, 0]  # (B, T_test)
        others_all = Y_test_predict[2: , :, :args.N_test, 0]  # others: (E, B, T)
        target = Y_test[:, :, 0]
        total_mse = (base_all - target).pow(2).mean()
        Loss.append(total_mse.item())

        base_expand = base_all.unsqueeze(0).expand_as(others_all)
        score_EN = Causal_score(
        others_all,
        base_expand,
        method=method,
        tau=tau)# (E, B, T)
        Score = score_EN.transpose(0, 1).contiguous()  # (B, E)
        Score = shift_column_for_Causal_Matrix(Score)
        Score = Score.detach().cpu()
        Score_all.append(Score)
        if st == "01":
            Score_01.append(Score)
        else:
            Score_02.append(Score)

mean_loss = np.mean(Loss)
print(f"Average Loss: {mean_loss:.6f}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def row_high_order_ratio(Score, n=7, ignore_diag=True):
    """
    对一个 Score 矩阵，计算每个 target 脑区的高阶因果占比。
    返回 shape: (n,)
    """
    S = to_numpy(Score).copy()

    if ignore_diag:
        for i in range(n):
            S[i, i] = 0.0

    pair = S[:, :n].sum(axis=1)
    higher = S[:, n:].sum(axis=1)

    rho = higher / (pair + higher + 1e-12)
    return rho


def plot_scoreall_percentile_violin(Score_all, labels=None, n=7):
    """
    Score_all: list of Score matrices
               each Score shape = (n, n + C(n-1,2))
    labels: 可选，例如 ["Subject 1 Run 1", ...]
    """

    # 每个被试/Run 得到 n 个 target 的 rho
    rho_list = [row_high_order_ratio(S, n=n) for S in Score_all]

    # 所有 rho 合并
    rho_all = np.concatenate(rho_list)

    # 每个被试/Run 的中位数
    medians = np.array([np.median(rho) for rho in rho_list])

    percentiles = [5, 35, 65, 95]
    selected_idx = []

    for p in percentiles:
        target_value = np.percentile(medians, p)
        idx = np.argmin(np.abs(medians - target_value))
        selected_idx.append(idx)

    data_plot = [rho_all] + [rho_list[idx] for idx in selected_idx]

    xlabels = ["All"] + [f"{p}%" for p in percentiles]

    plt.figure(figsize=(9, 4))

    parts = plt.violinplot(
        data_plot,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    # 美化 violin
    colors = ["#3E3A5E", "#C65392", "#EF5C82", "#FF7F73", "#F4A272"]

    for pc, c in zip(parts["bodies"], colors):
        pc.set_facecolor(c)
        pc.set_edgecolor("black")
        pc.set_alpha(0.85)

    # 加中位数和四分位数
    for i, vals in enumerate(data_plot, start=1):
        q1, med, q3 = np.percentile(vals, [25, 50, 75])

        plt.scatter(i, med, color="black", s=45, zorder=3)
        plt.vlines(i, q1, q3, color="black", linewidth=6, zorder=2)
        plt.vlines(i, vals.min(), vals.max(), color="black", linewidth=1.5, zorder=1)

    plt.xticks(range(1, len(xlabels) + 1), xlabels, fontsize=12)
    plt.ylabel(r"$\rho$", fontsize=14)
    plt.xlabel("Percentile", fontsize=14)
    plt.ylim(0, 1.05)
    plt.grid(axis="y", alpha=0.25)

    # 标注代表被试/Run
    if labels is not None:
        for xpos, idx in enumerate(selected_idx, start=2):
            label = labels[idx]
            y = np.max(rho_list[idx]) + 0.05
            plt.text(
                xpos,
                min(y, 1.0),
                label,
                ha="center",
                va="bottom",
                fontsize=10
            )

    plt.tight_layout()
    plt.show()

    return {
        "rho_list": rho_list,
        "rho_all": rho_all,
        "medians": medians,
        "selected_idx": selected_idx,
        "selected_percentiles": percentiles,
    }
out = plot_scoreall_percentile_violin(
    Score_all,
    labels=None,   # 如果你有 ["Subject #.. Run #.."] 可以传进去
    n=7
)

In [ ]:
from collections import Counter
import numpy as np
def top10_freq_per_sequence(Ainf_list, order=3, top_n=100):
    all_top_edges = []

    for seq_idx, Ainf in enumerate(Ainf_list):
        if order not in Ainf or Ainf[order].shape[0] == 0:
            continue

        edges = Ainf[order]
        weights = np.abs(edges[:, -1])
        idx_top = np.argsort(weights)[-top_n:][::-1]
        top_edges = edges[idx_top, :order]
        all_top_edges.extend([tuple(row.astype(int)) for row in top_edges])

    counter = Counter(all_top_edges)
    return counter

Ainf_PIRC_list=[build_T_to_Ainf(Score.detach().cpu().numpy(),3) for Score in Score_all]

top_select=5

freq_PIRC = top10_freq_per_sequence(Ainf_PIRC_list, order=3, top_n=top_select)

print(f"PIRC:{freq_PIRC.most_common()}")

In [ ]:
from collections import Counter
import numpy as np
import math

def top_percent_freq_per_sequence(Ainf_list, order=3, top_percent=0.10):
    all_top_edges = []

    for seq_idx, Ainf in enumerate(Ainf_list):
        if order not in Ainf or Ainf[order].shape[0] == 0:
            continue

        edges = Ainf[order]
        weights = np.abs(edges[:, -1])

        M = len(weights)
        k = max(1, int(np.ceil(top_percent * M)))

        idx_top = np.argsort(weights)[-k:][::-1]

        for row in edges[idx_top]:
            idx = row[:order].astype(int)

            if order == 3:
                target = int(idx[0])
                j, k0 = sorted([int(idx[1]), int(idx[2])])
                edge = (target, j, k0)
            else:
                edge = tuple(idx)

            all_top_edges.append(edge)

    counter = Counter(all_top_edges)
    return counter
Ainf_PIRC_list = [
    build_T_to_Ainf(Score.detach().cpu().numpy(), 3)
    for Score in Score_all
]

freq_PIRC = top_percent_freq_per_sequence(
    Ainf_PIRC_list,
    order=3,
    top_percent=0.10
)

print("PIRC top 10% hyperedges:")
print(freq_PIRC.most_common())
n_seq = len(Ainf_PIRC_list)

freq_ratio = {
    edge: count / n_seq
    for edge, count in freq_PIRC.items()
}

freq_ratio_sorted = sorted(
    freq_ratio.items(),
    key=lambda x: x[1],
    reverse=True
)

print(freq_ratio_sorted)

In [ ]:
import numpy as np
def high_order_ratio(Score, n=7):
    if hasattr(Score, "detach"):
        Score = Score.detach().cpu().numpy()
    Score = np.asarray(Score).copy()
    for i in range(n):
        Score[i, i] = 0.0
    higher = Score[:, n:].sum()
    total = Score.sum()
    return higher / (total + 1e-12)

In [ ]:
# rho_rest = np.array([high_order_ratio(S, n=7) for S in Score_rest])
# rho_action = np.array([high_order_ratio(S, n=7) for S in Score_action])
rho_all = np.array([high_order_ratio(S, n=7) for S in Score_all])

# print("Rest:", rho_rest.mean(), rho_rest.std())
# print("Action:", rho_action.mean(), rho_action.std())
print("All:", rho_all.mean(), rho_all.std())

plt.figure(figsize=(5, 4))

data_plot = [rho_all]

plt.violinplot(
    data_plot,
    showmeans=True,
    showmedians=True,
    showextrema=True
)

plt.xticks([1], ["All"])
plt.ylabel("Higher-order causality ratio")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.cluster import KMeans


def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def stack_score_all(Score_all):
    return np.concatenate([to_numpy(s) for s in Score_all], axis=0)


def plot_global_kmeans_threshold_jitter(
    Score_all,
    n=7,
    trim_percent=5,
    remove_diag=True,
    save_path="Global_KMeans_threshold_jitter.png",
    random_state=0,
):
    score = stack_score_all(Score_all)

    mask = np.ones_like(score, dtype=bool)

    if remove_diag:
        # 只去掉 pairwise 部分的自连接
        for i in range(n):
            mask[i::n, i] = False

    x_raw = score[mask]
    x_raw = x_raw[np.isfinite(x_raw)]

    low = np.percentile(x_raw, trim_percent)
    high = np.percentile(x_raw, 100 - trim_percent)

    x_trim = x_raw[(x_raw >= low) & (x_raw <= high)]

    km = KMeans(
        n_clusters=2,
        random_state=random_state,
        n_init=20
    )

    labels_trim = km.fit_predict(x_trim.reshape(-1, 1))
    centers = km.cluster_centers_.flatten()

    low_id, high_id = np.argsort(centers)
    c_low, c_high = np.sort(centers)

    threshold = (c_low + c_high) / 2

    no_causal = labels_trim == low_id
    causal = labels_trim == high_id

    range_left = x_trim[no_causal].max()
    range_right = x_trim[causal].min()

    rng = np.random.default_rng(random_state)
    jitter = rng.normal(0, 1.25, size=len(x_trim))

    fig, ax = plt.subplots(figsize=(5, 4), dpi=1200)

    ax.axvspan(
        range_left,
        range_right,
        color="#f4a340",
        alpha=0.18,
        linewidth=0,
        label="Feasible threshold range"
    )

    ax.scatter(
        x_trim[no_causal],
        jitter[no_causal],
        s=45,
        facecolors="white",
        edgecolors="0.72",
        linewidths=1.2,
        alpha=0.85,
        label="No causal effect"
    )

    ax.scatter(
        x_trim[causal],
        jitter[causal],
        s=45,
        color="#1f77b4",
        edgecolors="#1f77b4",
        linewidths=0.6,
        alpha=0.85,
        label="Causal effect exists"
    )

    ax.axvline(
        threshold,
        color="red",
        linestyle="--",
        linewidth=2.0,
        label="K-means threshold"
    )

    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))
    ax.xaxis.set_major_formatter(formatter)
    ax.xaxis.get_offset_text().set_size(13)

    ax.set_xlabel("CI", fontsize=16)
    ax.set_yticks([])
    ax.grid(False)

    ax.tick_params(axis="both", labelsize=13, width=1.2, length=5)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color("black")
        ax.spines[spine].set_linewidth(1.2)

    handles, labels_ = ax.get_legend_handles_labels()
    order_names = [
        "No causal effect",
        "Causal effect exists",
        "K-means threshold",
        "Feasible threshold range",
    ]
    order = [labels_.index(name) for name in order_names if name in labels_]

    ax.legend(
        [handles[i] for i in order],
        [labels_[i] for i in order],
        fontsize=11,
        frameon=True,
        loc="upper right"
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()

    print(f"Raw score range: [{x_raw.min():.6e}, {x_raw.max():.6e}]")
    print(f"Trim range ({trim_percent}%-{100-trim_percent}%): [{low:.6e}, {high:.6e}]")
    print("Cluster centers:", np.sort(centers))
    print(f"K-means threshold: {threshold:.6e}")
    print(f"Feasible threshold range: [{range_left:.6e}, {range_right:.6e}]")

    return {
        "threshold": threshold,
        "feasible_range": (range_left, range_right),
        "centers": np.sort(centers),
        "trim_range": (low, high),
        "x_trim": x_trim,
        "labels_trim": labels_trim,
    }
out = plot_global_kmeans_threshold_jitter(
    Score_all,
    n=7,
    trim_percent=5,
    remove_diag=True,
    save_path="Global_KMeans_threshold_jitter.png",
    random_state=0
)

threshold = out["threshold"]

In [ ]:
from collections import Counter
import numpy as np

def top_freq_per_sequence(
    Ainf_list,
    threshold,
    order=3,
    top_n=5,
):
    """
    统计每个序列中:
        1. 去掉 |weight| < threshold 的边
        2. 在剩余边中取 Top N
        3. 统计出现频率
    """

    all_top_edges = []

    for Ainf in Ainf_list:

        if order not in Ainf or len(Ainf[order]) == 0:
            continue

        edges = Ainf[order]

        # 权重
        weights = np.abs(edges[:, -1])

        # ---------- 去掉低于阈值 ----------
        valid = weights >= threshold

        if np.sum(valid) == 0:
            continue

        edges = edges[valid]
        weights = weights[valid]

        # ---------- Top N ----------
        k = min(top_n, len(weights))

        idx_top = np.argsort(weights)[-k:][::-1]

        top_edges = edges[idx_top, :order]

        all_top_edges.extend(
            [tuple(row.astype(int)) for row in top_edges]
        )

    return Counter(all_top_edges)

Ainf_PIRC_list = [
    build_T_to_Ainf(Score.detach().cpu().numpy(), 3)
    for Score in Score_all
]

freq_PIRC = top_freq_per_sequence(
    Ainf_PIRC_list,
    threshold=0.1,   # KMeans得到的阈值
    order=3,
    top_n=5
)

print(freq_PIRC.most_common())

In [ ]:
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from scipy.stats import mannwhitneyu
#
#
# # =========================
# # Basic tools
# # =========================
#
# def to_numpy(x):
#     if isinstance(x, torch.Tensor):
#         return x.detach().cpu().numpy()
#     return np.asarray(x)
#
#
# def pairwise_strength(Score, n=7):
#     Score = to_numpy(Score).copy()
#
#     # 去掉 pairwise 对角线
#     for i in range(n):
#         Score[i, i] = 0.0
#
#     pair = Score[:, :n]
#
#     mask = np.ones_like(pair, dtype=bool)
#     np.fill_diagonal(mask, False)
#
#     return pair[mask].mean()
#
#
# def higher_order_strength(Score, n=7):
#     Score = to_numpy(Score).copy()
#
#     higher = Score[:, n:]
#
#     return higher.mean()
#
#
# def high_order_ratio(Score, n=7):
#     Score = to_numpy(Score).copy()
#
#     for i in range(n):
#         Score[i, i] = 0.0
#
#     pair = Score[:, :n].sum()
#     higher = Score[:, n:].sum()
#
#     return higher / (pair + higher + 1e-12)
#
#
# # =========================
# # Compute statistics
# # =========================
#
# pair_rest = np.array([pairwise_strength(S, n=7) for S in Score_rest])
# pair_action = np.array([pairwise_strength(S, n=7) for S in Score_action])
# pair_all = np.array([pairwise_strength(S, n=7) for S in Score_all])
#
# higher_rest = np.array([higher_order_strength(S, n=7) for S in Score_rest])
# higher_action = np.array([higher_order_strength(S, n=7) for S in Score_action])
# higher_all = np.array([higher_order_strength(S, n=7) for S in Score_all])
#
# ratio_rest = np.array([high_order_ratio(S, n=7) for S in Score_rest])
# ratio_action = np.array([high_order_ratio(S, n=7) for S in Score_action])
# ratio_all = np.array([high_order_ratio(S, n=7) for S in Score_all])
#
#
# # =========================
# # Print summary
# # =========================
#
# def print_summary(name, rest, action):
#     stat, p = mannwhitneyu(rest, action, alternative="two-sided")
#     print(f"\n{name}")
#     print(f"Rest   : {rest.mean():.4f} ± {rest.std():.4f}")
#     print(f"Action : {action.mean():.4f} ± {action.std():.4f}")
#     print(f"Mann-Whitney p = {p:.4e}")
#
#
# print_summary("Pairwise strength", pair_rest, pair_action)
# print_summary("Higher-order strength", higher_rest, higher_action)
# print_summary("Higher-order ratio", ratio_rest, ratio_action)
#
#
# # =========================
# # Plot 2x2 figure
# # =========================
#
# fig, axes = plt.subplots(2, 2, figsize=(9, 7))
#
# # ---- Pairwise strength ----
# axes[0, 0].violinplot(
#     [pair_rest, pair_action, pair_all],
#     showmeans=True,
#     showmedians=True,
#     showextrema=True
# )
# axes[0, 0].set_xticks([1, 2, 3])
# axes[0, 0].set_xticklabels(["Rest", "Action", "All"])
# axes[0, 0].set_ylabel("Pairwise strength")
# axes[0, 0].set_title("Pairwise strength")
# axes[0, 0].grid(axis="y", alpha=0.3)
#
# # ---- Higher-order strength ----
# axes[0, 1].violinplot(
#     [higher_rest, higher_action, higher_all],
#     showmeans=True,
#     showmedians=True,
#     showextrema=True
# )
# axes[0, 1].set_xticks([1, 2, 3])
# axes[0, 1].set_xticklabels(["Rest", "Action", "All"])
# axes[0, 1].set_ylabel("Higher-order strength")
# axes[0, 1].set_title("Higher-order strength")
# axes[0, 1].grid(axis="y", alpha=0.3)
#
# # ---- Higher-order ratio ----
# axes[1, 0].violinplot(
#     [ratio_rest, ratio_action, ratio_all],
#     showmeans=True,
#     showmedians=True,
#     showextrema=True
# )
# axes[1, 0].set_xticks([1, 2, 3])
# axes[1, 0].set_xticklabels(["Rest", "Action", "All"])
# axes[1, 0].set_ylabel("Higher-order causality ratio")
# axes[1, 0].set_title("Higher-order ratio")
# axes[1, 0].grid(axis="y", alpha=0.3)
#
# # ---- Pairwise vs Higher-order scatter ----
# axes[1, 1].scatter(
#     pair_rest,
#     higher_rest,
#     alpha=0.7,
#     label="Rest"
# )
# axes[1, 1].scatter(
#     pair_action,
#     higher_action,
#     alpha=0.7,
#     label="Action"
# )
#
# axes[1, 1].set_xlabel("Pairwise strength")
# axes[1, 1].set_ylabel("Higher-order strength")
# axes[1, 1].set_title("Pairwise vs higher-order")
# axes[1, 1].legend(frameon=False)
# axes[1, 1].grid(alpha=0.3)
#
# plt.tight_layout()
# plt.show()

In [ ]:
# def target_high_ratio(Score, target, n=7):
#     if hasattr(Score, "detach"):
#         Score = Score.detach().cpu().numpy()
#     for i in range(n):
#         Score[i, i] = 0.0
#     row = Score[target]
#     higher = row[n:].sum()
#     total = row.sum()
#     return higher / (total + 1e-12)
#
# target = 0
#
# rho01 = [target_high_ratio(S, target) for S in Score_rest]
# rho02 = [target_high_ratio(S, target) for S in Score_action]
# rho_all = [target_high_ratio(S, target) for S in Score_all]
#
# plt.figure(figsize=(5, 4))
#
# data_plot = [rho01, rho02, rho_all]
#
# plt.violinplot(
#     data_plot,
#     showmeans=True,
#     showmedians=True,
#     showextrema=True
# )
#
# plt.xticks([1, 2, 3], ["Rest R01", "Rest R02", "All"])
# plt.ylabel("Higher-order causality ratio")
# plt.grid(axis="y", alpha=0.3)
#
# plt.tight_layout()
# plt.show()

In [ ]:
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from scipy.stats import wilcoxon
#
# # =========================
# # 1. 工具函数
# # =========================
#
# def to_numpy(x):
#     if isinstance(x, torch.Tensor):
#         return x.detach().cpu().numpy()
#     return np.asarray(x)
#
#
# def high_order_ratio_global(Score, n=7, ignore_diag=True):
#     """
#     整个网络的高阶因果占比
#     Score: shape (n, n + C(n-1,2))
#     """
#     Score = to_numpy(Score).copy()
#
#     if ignore_diag:
#         for i in range(n):
#             Score[i, i] = 0.0
#
#     higher = Score[:, n:].sum()
#     total = Score.sum()
#
#     return higher / (total + 1e-12)
#
#
# def high_order_ratio_target(Score, target, n=7, ignore_self=True):
#     """
#     某个 target 脑区的高阶因果占比
#     """
#     Score = to_numpy(Score).copy()
#
#     row = Score[target, :].copy()
#
#     if ignore_self:
#         row[target] = 0.0
#
#     higher = row[n:].sum()
#     total = row.sum()
#
#     return higher / (total + 1e-12)
#
#
# # =========================
# # 2. 按被试计算全局占比
# # =========================
#
# rho_01 = np.array([
#     high_order_ratio_global(S, n=7)
#     for S in Score_rest
# ])
#
# rho_02 = np.array([
#     high_order_ratio_global(S, n=7)
#     for S in Score_action
# ])
#
# rho_all = np.array([
#     high_order_ratio_global(S, n=7)
#     for S in Score_all
# ])
#
# print("Global higher-order ratio")
# print("R01 mean ± std:", rho_01.mean(), rho_01.std())
# print("R02 mean ± std:", rho_02.mean(), rho_02.std())
# print("All mean ± std:", rho_all.mean(), rho_all.std())
#
# # stat, p = wilcoxon(rho_01, rho_02)
# # print("Global R01 vs R02 Wilcoxon p =", p)
#
#
# # =========================
# # 3. 全局 violin 图
# # =========================
#
# plt.figure(figsize=(5, 4))
#
# plt.violinplot(
#     [rho_01, rho_02, rho_all],
#     showmeans=True,
#     showmedians=True,
#     showextrema=True
# )
#
# plt.xticks([1, 2, 3], ["Rest R01", "Rest R02", "All"])
# plt.ylabel("Higher-order causality ratio")
# plt.grid(axis="y", alpha=0.3)
#
# plt.tight_layout()
# plt.show()
#
#
# # =========================
# # 4. 每个 target 脑区计算占比
# # =========================
#
# n = 7
#
# rho_01_target = np.zeros((len(Score_rest), n))
# rho_02_target = np.zeros((len(Score_action), n))
#
# for s_idx, S in enumerate(Score_rest):
#     for target in range(n):
#         rho_01_target[s_idx, target] = high_order_ratio_target(
#             S, target, n=n
#         )
#
# for s_idx, S in enumerate(Score_action):
#     for target in range(n):
#         rho_02_target[s_idx, target] = high_order_ratio_target(
#             S, target, n=n
#         )
#
#
# # =========================
# # 5. 每个脑区统计检验
# # =========================
#
# print("\nTarget-wise higher-order ratio")
# p_values = []
#
# for target in range(n):
#     mean_01 = rho_01_target[:, target].mean()
#     std_01 = rho_01_target[:, target].std()
#
#     mean_02 = rho_02_target[:, target].mean()
#     std_02 = rho_02_target[:, target].std()
#
#     # stat, p = wilcoxon(
#     #     rho_01_target[:, target],
#     #     rho_02_target[:, target]
#     # )
#     #
#     # p_values.append(p)
#
#     print(
#         f"Target {target}: "
#         f"R01={mean_01:.4f}±{std_01:.4f}, "
#         f"R02={mean_02:.4f}±{std_02:.4f}, "
#         # f"p={p:.4e}"
#     )
#
# # p_values = np.array(p_values)
#
#
# # =========================
# # 6. 每个脑区 violin 图
# # =========================
#
# zone_names = [f"Zone {i}" for i in range(n)]
#
# fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 4), sharey=True)
#
# for target in range(n):
#     ax = axes[target]
#
#     ax.violinplot(
#         [
#             rho_01_target[:, target],
#             rho_02_target[:, target]
#         ],
#         showmeans=True,
#         showmedians=True,
#         showextrema=True
#     )
#
#     ax.set_title(zone_names[target])
#     ax.set_xticks([1, 2])
#     ax.set_xticklabels(["R01", "R02"], rotation=30)
#     ax.grid(axis="y", alpha=0.3)
#
#     # if p_values[target] < 0.001:
#     #     sig = "***"
#     # elif p_values[target] < 0.01:
#     #     sig = "**"
#     # elif p_values[target] < 0.05:
#     #     sig = "*"
#     # else:
#     #     sig = "n.s."
#
#     ymax = max(
#         rho_01_target[:, target].max(),
#         rho_02_target[:, target].max()
#     )
#
#     # ax.text(
#     #     1.5,
#     #     ymax + 0.03,
#     #     # sig,
#     #     ha="center",
#     #     va="bottom",
#     #     fontsize=12
#     # )
#
# axes[0].set_ylabel("Higher-order causality ratio")
#
# plt.tight_layout()
# plt.show()

In [ ]:
# def plot_subject_percentile_mixed(
#     Score_01,
#     Score_02,
#     subject_labels=None,
#     n=7,
#     seed=42
# ):
#     assert len(Score_01) == len(Score_02)
#
#     rng = np.random.default_rng(seed)
#
#     rho_subject = []
#
#     for S01, S02 in zip(Score_01, Score_02):
#         rho01 = high_order_ratio_by_target(S01, n=n)
#         rho02 = high_order_ratio_by_target(S02, n=n)
#
#         rho = np.concatenate([rho01, rho02])  # 14 values
#         rho_subject.append(rho)
#
#     rho_all = np.concatenate(rho_subject)
#
#     medians = np.array([np.median(rho) for rho in rho_subject])
#
#     percentiles = [5, 35, 65, 95]
#     selected_idx = []
#
#     for p in percentiles:
#         target_value = np.percentile(medians, p)
#         idx = np.argmin(np.abs(medians - target_value))
#         selected_idx.append(idx)
#
#     data_plot = [rho_all] + [rho_subject[idx] for idx in selected_idx]
#     xlabels = ["All"] + [f"{p}%" for p in percentiles]
#
#     fig, ax = plt.subplots(figsize=(9, 4))
#
#     # 1) All 用 violin
#     parts = ax.violinplot(
#         [rho_all],
#         positions=[1],
#         showmeans=False,
#         showmedians=False,
#         showextrema=True
#     )
#
#     parts["bodies"][0].set_facecolor("#3E3A5E")
#     parts["bodies"][0].set_edgecolor("black")
#     parts["bodies"][0].set_alpha(0.85)
#
#     # 2) 代表受试者用 boxplot
#     box = ax.boxplot(
#         data_plot[1:],
#         positions=[2, 3, 4, 5],
#         widths=0.25,
#         patch_artist=True,
#         showfliers=False
#     )
#
#     colors = ["#C65392", "#EF5C82", "#FF7F73", "#F4A272"]
#
#     for patch, c in zip(box["boxes"], colors):
#         patch.set_facecolor(c)
#         patch.set_alpha(0.6)
#         patch.set_edgecolor("black")
#
#     for key in ["whiskers", "caps", "medians"]:
#         for item in box[key]:
#             item.set_color("black")
#             item.set_linewidth(1.5)
#
#     # 3) 加 jitter scatter
#     for xpos, vals, c in zip([2, 3, 4, 5], data_plot[1:], colors):
#         x_jitter = rng.normal(xpos, 0.035, size=len(vals))
#         ax.scatter(
#             x_jitter,
#             vals,
#             s=28,
#             alpha=0.75,
#             color=c,
#             edgecolor="black",
#             linewidth=0.3,
#             zorder=3
#         )
#
#     # 4) All 的中位数和四分位数
#     q1, med, q3 = np.percentile(rho_all, [25, 50, 75])
#     ax.scatter(1, med, color="black", s=45, zorder=4)
#     ax.vlines(1, q1, q3, color="black", linewidth=6, zorder=3)
#
#     ax.set_xticks([1, 2, 3, 4, 5])
#     ax.set_xticklabels(xlabels)
#     ax.set_xlabel("Percentile")
#     ax.set_ylabel(r"$\rho$")
#     ax.set_ylim(0, 1.05)
#     ax.grid(axis="y", alpha=0.25)
#
#     if subject_labels is not None:
#         for xpos, idx in enumerate(selected_idx, start=2):
#             ax.text(
#                 xpos,
#                 0.98,
#                 subject_labels[idx],
#                 ha="center",
#                 va="top",
#                 fontsize=10
#             )
#
#     plt.tight_layout()
#     plt.show()
#
#     return {
#         "rho_subject": rho_subject,
#         "rho_all": rho_all,
#         "medians": medians,
#         "selected_idx": selected_idx,
#         "selected_percentiles": percentiles,
#     }
#


In [ ]:
# import numpy as np
# import torch
# import matplotlib.pyplot as plt
# from sklearn.cluster import KMeans
# from scipy.stats import wilcoxon
#
#
# def to_numpy(x):
#     if isinstance(x, torch.Tensor):
#         return x.detach().cpu().numpy()
#     return np.asarray(x)
#
#
# def get_global_kmeans_threshold(Score_all, n=7, ignore_diag=True):
#     """
#     用所有被试、所有状态的 Score 全局估计 KMeans 阈值。
#     """
#
#     values = []
#
#     for S in Score_all:
#         S = to_numpy(S).copy()
#
#         if ignore_diag:
#             for i in range(n):
#                 S[i, i] = 0.0
#
#         values.append(S.reshape(-1))
#
#     values = np.concatenate(values)
#     values = values[np.isfinite(values)]
#     values = values[values > 0]
#
#     km = KMeans(
#         n_clusters=2,
#         random_state=0,
#         n_init=20
#     ).fit(values.reshape(-1, 1))
#
#     centers = np.sort(km.cluster_centers_.flatten())
#     threshold = centers.mean()
#
#     return threshold, centers
#
#
# def threshold_score(Score, threshold, n=7, ignore_diag=True):
#     """
#     根据全局阈值筛选有效因果。
#     """
#     S = to_numpy(Score).copy()
#
#     if ignore_diag:
#         for i in range(n):
#             S[i, i] = 0.0
#
#     S_thr = S.copy()
#     S_thr[S_thr < threshold] = 0.0
#
#     return S_thr
#
#
# def high_order_ratio_global_thresholded(Score, threshold, n=7):
#     """
#     全局高阶因果占比：
#     higher / (pairwise + higher)
#     """
#     S = threshold_score(Score, threshold, n=n)
#
#     pair = S[:, :n].sum()
#     higher = S[:, n:].sum()
#
#     return higher / (pair + higher + 1e-12)
#
#
# def high_order_ratio_target_thresholded(Score, threshold, target, n=7):
#     """
#     某个 target 脑区的高阶因果占比。
#     """
#     S = threshold_score(Score, threshold, n=n)
#
#     row = S[target, :]
#
#     pair = row[:n].sum()
#     higher = row[n:].sum()
#
#     return higher / (pair + higher + 1e-12)
#
#
# # =========================
# # 1. 全局 KMeans 阈值
# # =========================
#
# threshold, centers = get_global_kmeans_threshold(
#     Score_all,
#     n=7,
#     ignore_diag=True
# )
#
# print("KMeans centers:", centers)
# print("Global threshold:", threshold)
#
#
# # =========================
# # 2. 全局高阶比例
# # =========================
#
# rho_01 = np.array([
#     high_order_ratio_global_thresholded(S, threshold, n=7)
#     for S in Score_01
# ])
#
# rho_02 = np.array([
#     high_order_ratio_global_thresholded(S, threshold, n=7)
#     for S in Score_02
# ])
#
# rho_all = np.array([
#     high_order_ratio_global_thresholded(S, threshold, n=7)
#     for S in Score_all
# ])
#
# print("\nGlobal thresholded higher-order ratio")
# print(f"R01 mean ± std: {rho_01.mean():.4f} ± {rho_01.std():.4f}")
# print(f"R02 mean ± std: {rho_02.mean():.4f} ± {rho_02.std():.4f}")
# print(f"All mean ± std: {rho_all.mean():.4f} ± {rho_all.std():.4f}")
#
# stat, p = wilcoxon(rho_01, rho_02)
# print(f"Global R01 vs R02 Wilcoxon p = {p:.4e}")
#
#
# # =========================
# # 3. 全局 violin 图
# # =========================
#
# plt.figure(figsize=(5, 4))
#
# plt.violinplot(
#     [rho_01, rho_02, rho_all],
#     showmeans=True,
#     showmedians=True,
#     showextrema=True
# )
#
# plt.xticks([1, 2, 3], ["Rest R01", "Rest R02", "All"])
# plt.ylabel("Thresholded higher-order causality ratio")
# plt.grid(axis="y", alpha=0.3)
# plt.tight_layout()
# plt.show()
#
#
# # =========================
# # 4. 每个 target 脑区高阶比例
# # =========================
#
# n = 7
#
# rho_01_target = np.zeros((len(Score_01), n))
# rho_02_target = np.zeros((len(Score_02), n))
#
# for s_idx, S in enumerate(Score_01):
#     for target in range(n):
#         rho_01_target[s_idx, target] = high_order_ratio_target_thresholded(
#             S, threshold, target, n=n
#         )
#
# for s_idx, S in enumerate(Score_02):
#     for target in range(n):
#         rho_02_target[s_idx, target] = high_order_ratio_target_thresholded(
#             S, threshold, target, n=n
#         )
#
#
# # =========================
# # 5. 每个脑区统计检验
# # =========================
#
# p_values = []
#
# print("\nTarget-wise thresholded higher-order ratio")
#
# for target in range(n):
#     stat, p = wilcoxon(
#         rho_01_target[:, target],
#         rho_02_target[:, target]
#     )
#
#     p_values.append(p)
#
#     print(
#         f"Target {target}: "
#         f"R01={rho_01_target[:, target].mean():.4f}±{rho_01_target[:, target].std():.4f}, "
#         f"R02={rho_02_target[:, target].mean():.4f}±{rho_02_target[:, target].std():.4f}, "
#         f"p={p:.4e}"
#     )
#
# p_values = np.array(p_values)
#
#
# # =========================
# # 6. 每个脑区 violin 图
# # =========================
#
# fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 4), sharey=True)
#
# for target in range(n):
#     ax = axes[target]
#
#     ax.violinplot(
#         [
#             rho_01_target[:, target],
#             rho_02_target[:, target]
#         ],
#         showmeans=True,
#         showmedians=True,
#         showextrema=True
#     )
#
#     ax.set_title(f"Zone {target}")
#     ax.set_xticks([1, 2])
#     ax.set_xticklabels(["R01", "R02"], rotation=30)
#     ax.grid(axis="y", alpha=0.3)
#
#     if p_values[target] < 0.001:
#         sig = "***"
#     elif p_values[target] < 0.01:
#         sig = "**"
#     elif p_values[target] < 0.05:
#         sig = "*"
#     else:
#         sig = "n.s."
#
#     ymax = max(
#         rho_01_target[:, target].max(),
#         rho_02_target[:, target].max()
#     )
#
#     ax.text(
#         1.5,
#         ymax + 0.03,
#         sig,
#         ha="center",
#         va="bottom",
#         fontsize=12
#     )
#
# axes[0].set_ylabel("Thresholded higher-order causality ratio")
#
# plt.tight_layout()
# plt.show()